In [6]:
import pandas as pd
import numpy as np
df=pd.read_csv('delhivery_data.csv')

In [7]:
df.head()

,data,trip_creation_time,route_schedule_uuid,route_type,trip_uuid,source_center,source_name,destination_center,destination_name,od_start_time,...,cutoff_timestamp,actual_distance_to_destination,actual_time,osrm_time,osrm_distance,factor,segment_actual_time,segment_osrm_time,segment_osrm_distance,segment_factor
0,training,2018-09-20 02:35:36.476840,thanos::sroute:eb7bfc78-b351-4c0e-a951-fa3d5c3...,Carting,trip-153741093647649320,IND388121AAA,Anand_VUNagar_DC (Gujarat),IND388620AAB,Khambhat_MotvdDPP_D (Gujarat),2018-09-20 03:21:32.418600,...,2018-09-20 04:27:55,10.435660,14.0,11.0,11.9653,1.272727,14.0,11.0,11.9653,1.272727
1,training,2018-09-20 02:35:36.476840,thanos::sroute:eb7bfc78-b351-4c0e-a951-fa3d5c3...,Carting,trip-153741093647649320,IND388121AAA,Anand_VUNagar_DC (Gujarat),IND388620AAB,Khambhat_MotvdDPP_D (Gujarat),2018-09-20 03:21:32.418600,...,2018-09-20 04:17:55,18.936842,24.0,20.0,21.7243,1.200000,10.0,9.0,9.7590,1.111111
2,training,2018-09-20 02:35:36.476840,thanos::sroute:eb7bfc78-b351-4c0e-a951-fa3d5c3...,Carting,trip-153741093647649320,IND388121AAA,Anand_VUNagar_DC (Gujarat),IND388620AAB,Khambhat_MotvdDPP_D (Gujarat),2018-09-20 03:21:32.418600,...,2018-09-20 04:01:19.505586,27.637279,40.0,28.0,32.5395,1.428571,16.0,7.0,10.8152,2.285714
3,training,2018-09-20 02:35:36.476840,thanos::sroute:eb7bfc78-b351-4c0e-a951-fa3d5c3...,Carting,trip-153741093647649320,IND388121AAA,Anand_VUNagar_DC (Gujarat),IND388620AAB,Khambhat_MotvdDPP_D (Gujarat),2018-09-20 03:21:32.418600,...,2018-09-20 03:39:57,36.118028,62.0,40.0,45.5620,1.550000,21.0,12.0,13.0224,1.750000
4,training,2018-09-20 02:35:36.476840,thanos::sroute:eb7bfc78-b351-4c0e-a951-fa3d5c3...,Carting,trip-153741093647649320,IND388121AAA,Anand_VUNagar_DC (Gujarat),IND388620AAB,Khambhat_MotvdDPP_D (Gujarat),2018-09-20 03:21:32.418600,...,2018-09-20 03:33:55,39.386040,68.0,44.0,54.2181,1.545455,6.0,5.0,3.9153,1.200000


In [8]:
df.info()
print(df.shape)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 144867 entries, 0 to 144866
Data columns (total 24 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   data                            144867 non-null  object 
 1   trip_creation_time              144867 non-null  object 
 2   route_schedule_uuid             144867 non-null  object 
 3   route_type                      144867 non-null  object 
 4   trip_uuid                       144867 non-null  object 
 5   source_center                   144867 non-null  object 
 6   source_name                     144574 non-null  object 
 7   destination_center              144867 non-null  object 
 8   destination_name                144606 non-null  object 
 9   od_start_time                   144867 non-null  object 
 10  od_end_time                     144867 non-null  object 
 11  start_scan_to_end_scan          144867 non-null  float64
 12  is_cutoff       

In [9]:
df.replace(['nan', 'NaN', 'NAN', 'n/a'], np.nan, inplace=True)
df.isnull().sum()

data                                0
trip_creation_time                  0
route_schedule_uuid                 0
route_type                          0
trip_uuid                           0
source_center                       0
source_name                       293
destination_center                  0
destination_name                  261
od_start_time                       0
od_end_time                         0
start_scan_to_end_scan              0
is_cutoff                           0
cutoff_factor                       0
cutoff_timestamp                    0
actual_distance_to_destination      0
actual_time                         0
osrm_time                           0
osrm_distance                       0
factor                              0
segment_actual_time                 0
segment_osrm_time                   0
segment_osrm_distance               0
segment_factor                      0
dtype: int64

In [11]:
df.dropna(subset=['source_name', 'destination_name'], inplace=True)
df.shape

(144316, 24)

In [12]:
date_cols = ['trip_creation_time', 'od_start_time', 'od_end_time']
for col in date_cols:
    df[col] = pd.to_datetime(df[col])

In [14]:
#Handling Outliers
df = df[df['actual_distance_to_destination'] > 0]

In [15]:
#extracting state from destination name
df['state'] = df['destination_name'].str.extract(r'\((.*?)\)')

In [16]:
#Handling cumulative data
cleaned_df = df.groupby(['trip_uuid', 'source_name', 'destination_name']).agg({
    'trip_creation_time': 'first',
    'state': 'first',
    'route_type': 'first',
    'actual_distance_to_destination': 'max', # Total distance
    'actual_time': 'max',                    # Total time taken
    'osrm_time': 'max',                      # Planned time
    'osrm_distance': 'max'                   # Planned distance
}).reset_index()

In [17]:
#Creating new features
cleaned_df['delay_minutes'] = cleaned_df['actual_time'] - cleaned_df['osrm_time']
cleaned_df['avg_speed'] = cleaned_df['actual_distance_to_destination'] / (cleaned_df['actual_time'] / 60)
cleaned_df['efficiency_ratio'] = cleaned_df['actual_time'] / cleaned_df['osrm_time']

In [18]:
print(f"Original Rows: {len(df)}")
print(f"Unique Trips: {len(cleaned_df)}")

cleaned_df.shape

Original Rows: 144316
Unique Trips: 26222


(26222, 13)

In [19]:
cleaned_df.head()

,trip_uuid,source_name,destination_name,trip_creation_time,state,route_type,actual_distance_to_destination,actual_time,osrm_time,osrm_distance,delay_minutes,avg_speed,efficiency_ratio
0,trip-153671041653548748,Bhopal_Trnsport_H (Madhya Pradesh),Kanpur_Central_H_6 (Uttar Pradesh),2018-09-12 00:00:16.535741,Uttar Pradesh,FTL,440.973689,830.0,394.0,544.8027,436.0,31.877616,2.106599
1,trip-153671041653548748,Kanpur_Central_H_6 (Uttar Pradesh),Gurgaon_Bilaspur_HB (Haryana),2018-09-12 00:00:16.535741,Haryana,FTL,383.759164,732.0,349.0,446.5496,383.0,31.455669,2.097421
2,trip-153671042288605164,Doddablpur_ChikaDPP_D (Karnataka),Chikblapur_ShntiSgr_D (Karnataka),2018-09-12 00:00:22.886430,Karnataka,Carting,24.644021,47.0,26.0,28.1994,21.0,31.460452,1.807692
3,trip-153671042288605164,Tumkur_Veersagr_I (Karnataka),Doddablpur_ChikaDPP_D (Karnataka),2018-09-12 00:00:22.886430,Karnataka,Carting,48.542890,96.0,42.0,56.9116,54.0,30.339306,2.285714
4,trip-153671043369099517,Bangalore_Nelmngla_H (Karnataka),Gurgaon_Bilaspur_HB (Haryana),2018-09-12 00:00:33.691250,Haryana,FTL,1689.964663,2736.0,1529.0,2090.8743,1207.0,37.060629,1.789405


In [21]:
cleaned_df.to_csv('delhivery_data_cleaned.csv', index=False)

In [22]:
pip install psycopg2-binary sqlalchemy

Note: you may need to restart the kernel to use updated packages.


In [23]:
from sqlalchemy import create_engine
from urllib.parse import quote_plus

# connect to postgreSQL
username = "postgres"
password = quote_plus("Jajaha271508@")
host = "localhost"        # or IP address
port = "5432"             # default PostgreSQL port
database = "logistics_customer_satisfaction"

engine = create_engine(
    f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}"
)

#load dataframe into postgreSQL
table_name="logistics"
cleaned_df.to_sql(table_name,engine,if_exists="replace",index=False)

print(f"data successfully loaded into tables'{table_name}'in database'{database}'.")

data successfully loaded into tables'logistics'in database'logistics_customer_satisfaction'.
